In [1]:
%pip install datasets pandas pyarrow


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: /Users/santhosh/myenv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset
import pandas as pd

# Grab the full dataset so we can see what we're working with
ds_full = load_dataset("ncbi/Open-Patients", split="train")
df_full = ds_full.to_pandas()

# Pull the source type out of the _id field
df_full['source'] = df_full['_id'].apply(lambda x: x.rsplit('-', 1)[0] if '-' in str(x) else 'unknown')

print(f"Total records: {len(df_full)}")
print(f"\nSource distribution:\n{df_full['source'].value_counts()}")

Total records: 180142

Source distribution:
source
usmle            12893
trec-ct-2021        75
trec-ct-2022        50
pmc-8391342         34
trec-cds-2015       30
                 ...  
pmc-8556118          1
pmc-8556096          1
pmc-8556093          1
pmc-8556051          1
pmc-4628016          1
Name: count, Length: 140903, dtype: int64


In [4]:
import re

def extract_source(id_str):
    id_str = str(id_str)
    if id_str.startswith('trec-cds'):
        return 'trec-cds'
    elif id_str.startswith('trec-ct'):
        return 'trec-ct'
    elif id_str.startswith('pmc'):
        return 'pmc'
    elif id_str.startswith('usmle'):
        return 'usmle'
    else:
        return 'other'

df_full['source'] = df_full['_id'].apply(extract_source)

print(f"Total records: {len(df_full)}")
print(f"\nSource distribution:\n{df_full['source'].value_counts()}")


Total records: 180142

Source distribution:
source
pmc         167034
usmle        12893
trec-ct        125
trec-cds        90
Name: count, dtype: int64


In [5]:
# Take ALL trec records (they're rare and valuable for comparison)
df_trec = df_full[df_full['source'].isin(['trec-ct', 'trec-cds'])]  # 215 records

# Sample remaining 785 proportionally from pmc and usmle
df_rest = df_full[~df_full['source'].isin(['trec-ct', 'trec-cds'])]
remaining_needed = 1000 - len(df_trec)

df_rest_sample = df_rest.groupby('source', group_keys=False).apply(
    lambda x: x.sample(n=int(remaining_needed * len(x) / len(df_rest)), random_state=42),
    include_groups=False
)
df_rest_sample = df_full.loc[df_rest_sample.index]

# Combine and adjust to exactly 1000
df_sample = pd.concat([df_trec, df_rest_sample])
if len(df_sample) < 1000:
    remaining = df_full.drop(df_sample.index)
    df_sample = pd.concat([df_sample, remaining.sample(1000 - len(df_sample), random_state=42)])

print(f"Sample size: {len(df_sample)}")
print(f"\nSample distribution:\n{df_sample['source'].value_counts()}")

Sample size: 1000

Sample distribution:
source
pmc         729
trec-ct     125
trec-cds     90
usmle        56
Name: count, dtype: int64


In [6]:
import numpy as np

# --- Length statistics ---
df_sample['char_len'] = df_sample['description'].astype(str).str.len()
df_sample['word_count'] = df_sample['description'].astype(str).str.split().str.len()
df_sample['approx_tokens'] = (df_sample['word_count'] * 1.3).astype(int)

print("=== OVERALL LENGTH STATISTICS ===")
print(df_sample[['char_len', 'word_count', 'approx_tokens']].describe().round(1))

# --- Per-source length comparison ---
print("\n=== PER-SOURCE WORD COUNT STATS ===")
print(df_sample.groupby('source')['word_count'].describe().round(1))

# --- Empty / garbage detection ---
empty = df_sample[df_sample['char_len'] < 20]
short = df_sample[(df_sample['char_len'] >= 20) & (df_sample['char_len'] < 100)]
print(f"\n=== QUALITY CHECK ===")
print(f"Records < 20 chars (empty/garbage): {len(empty)}")
print(f"Records 20-100 chars (suspiciously short): {len(short)}")
if len(empty) > 0:
    print("\nEmpty/garbage records:")
    print(empty[['_id', 'source', 'description']].to_string())
if len(short) > 0:
    print("\nShort records (first 5):")
    print(short[['_id', 'source', 'char_len', 'description']].head().to_string())

# --- Chunking analysis ---
needs_chunking = df_sample[df_sample['approx_tokens'] > 450]
print(f"\n=== CHUNKING ANALYSIS ===")
print(f"Records needing chunking (>450 tokens): {len(needs_chunking)} ({len(needs_chunking)/10:.1f}%)")
print(f"Breakdown by source:")
print(needs_chunking['source'].value_counts())

# --- Extremes ---
print(f"\n=== EXTREMES ===")
longest = df_sample.loc[df_sample['word_count'].idxmax()]
shortest = df_sample.loc[df_sample['word_count'].idxmin()]
print(f"Longest:  {longest['word_count']} words | source: {longest['source']} | id: {longest['_id']}")
print(f"Shortest: {shortest['word_count']} words | source: {shortest['source']} | id: {shortest['_id']}")

=== OVERALL LENGTH STATISTICS ===
       char_len  word_count  approx_tokens
count    1000.0      1000.0         1000.0
mean     2230.4       339.4          440.7
std      1681.1       253.3          329.2
min        69.0        14.0           18.0
25%       836.0       135.0          175.0
50%      1873.0       283.0          367.0
75%      3146.0       471.2          612.2
max     12558.0      1910.0         2483.0

=== PER-SOURCE WORD COUNT STATS ===
          count   mean    std   min    25%    50%    75%     max
source                                                          
pmc       729.0  424.5  246.2  14.0  254.0  384.0  542.0  1910.0
trec-cds   90.0   92.0   43.0  38.0   61.5   81.0  111.8   255.0
trec-ct   125.0  120.2   35.8  51.0   92.0  117.0  144.0   218.0
usmle      56.0  118.6   40.7  36.0   88.5  117.5  135.2   242.0

=== QUALITY CHECK ===
Records < 20 chars (empty/garbage): 0
Records 20-100 chars (suspiciously short): 1

Short records (first 5):
                  _i

In [7]:
# Let's eyeball a few records from each source to get a feel for the text
for source in ['pmc', 'usmle', 'trec-cds', 'trec-ct']:
    subset = df_sample[df_sample['source'] == source]
    print(f"\n{'='*70}")
    print(f"SOURCE: {source} | Count: {len(subset)} | Mean words: {subset['word_count'].mean():.0f}")
    print('='*70)
    
    # Pick one short, one mid-length, and one long example
    sorted_subset = subset.sort_values('word_count')
    indices = [0, len(sorted_subset)//2, -1]  # short, median, long
    
    for i, idx in enumerate(['SHORT', 'MEDIAN', 'LONG']):
        row = sorted_subset.iloc[indices[i]]
        print(f"\n--- {idx} ({row['word_count']} words) | ID: {row['_id']} ---")
        print(str(row['description'])[:600])
        if row['word_count'] > 90:
            print("... [truncated]")
        print()


SOURCE: pmc | Count: 729 | Mean words: 424

--- SHORT (14 words) | ID: pmc-8462249-2 ---
Case 2: A 75-year-old man had right groin swelling and pain for 6 mo.


--- MEDIAN (384 words) | ID: pmc-8033237-1 ---
A girl aged 9 years was investigated for continuous dribbling of urine since with recurrent urinary tract infection (UTI) and occasional pain in the left lumbar region. She also had a normal urge for urination with good stream. Physical examination revealed two urethral openings, one below the other in the sagittal plane, below the clitoris [Figure , and ] (Diagramatic representation)]. Her renal function tests (RFTs) were normal. Ultrasound scan of the abdomen showed gross hydroureteronephrosis of the left kidney extending up to the lower ureter and the urinary bladder had a very large capaci
... [truncated]


--- LONG (1910 words) | ID: pmc-3972028-1 ---
A 58-year-old woman presented to the Coburg Clinic in February 2013 with dyspnea, loss of energy, slight edema of the lower ex

In [11]:
# Save our sample so we don't have to re-download next time
import os
df_sample.to_csv("open_patients_1000_sample.csv", index=False)
print(f"Saved {len(df_sample)} records to open_patients_1000_sample.csv")
print(f"Columns: {list(df_sample.columns)}")
print(f"File size: {os.path.getsize('open_patients_1000_sample.csv') / 1024:.1f} KB")

Saved 1000 records to open_patients_1000_sample.csv
Columns: ['_id', 'description', 'source', 'char_len', 'word_count', 'approx_tokens']
File size: 2215.9 KB


In [12]:
print(f"Shape: {df_sample.shape}")
print(f"Columns: {list(df_sample.columns)}")
print(f"\nSource distribution:\n{df_sample['source'].value_counts()}")
print(f"\nSample records:")
print(df_sample[['_id', 'source', 'word_count']].head())

Shape: (1000, 6)
Columns: ['_id', 'description', 'source', 'char_len', 'word_count', 'approx_tokens']

Source distribution:
source
pmc         729
trec-ct     125
trec-cds     90
usmle        56
Name: count, dtype: int64

Sample records:
               _id    source  word_count
0  trec-cds-2014-1  trec-cds          99
1  trec-cds-2014-2  trec-cds          75
2  trec-cds-2014-3  trec-cds          48
3  trec-cds-2014-4  trec-cds         112
4  trec-cds-2014-5  trec-cds          78


In [14]:
import os

# Keep the version with profiling columns around for reference
df_sample.to_csv("open_patients_1000_profiled.csv", index=False)

# Slim version with just the columns the NER pipeline needs
df_sample[['_id', 'source', 'description']].to_csv("open_patients_1000_clean.csv", index=False)

print(f"Profiled version: {os.path.getsize('open_patients_1000_profiled.csv') / 1024:.1f} KB")
print(f"Clean version: {os.path.getsize('open_patients_1000_clean.csv') / 1024:.1f} KB")

Profiled version: 2215.9 KB
Clean version: 2203.6 KB
